In [0]:
!pip install ultralytics opencv-python -q

import torch

from ultralytics import YOLO, YOLOE
import cv2
import numpy as np
import logging
import json
from collections import defaultdict
import torch
import torch

class ArmMovementTracker:
    def __init__(self, movement_threshold=13, frame_memory=15):
        self.trackers = {}
        self.movement_threshold = movement_threshold
        self.frame_memory = frame_memory

    def get_tracker(self, person_id):
        if person_id not in self.trackers:
            self.trackers[person_id] = {
                'prev_keypoints': {'left': {'wrist': None, 'elbow': None}, 'right': {'wrist': None, 'elbow': None}},
                'movement_counter': {'left': 0, 'right': 0},
                'active_action': None,
                'keypoint_history': {'left': [], 'right': []},
                'action_active': {'left': False, 'right': False}
            }
        return self.trackers[person_id]

    def calculate_keypoint_movement(self, person_id, side, wrist_pos, elbow_pos, frame_count):
        tracker = self.get_tracker(person_id)
        prev_keypoints = tracker['prev_keypoints']
        history = tracker['keypoint_history'][side]
        
        if frame_count % 3 == 0:
            current_keypoints = {'wrist': wrist_pos, 'elbow': elbow_pos, 'frame': frame_count}
            history.append(current_keypoints)
            if len(history) > self.frame_memory:
                history.pop(0)
            prev_keypoints[side]['wrist'] = wrist_pos
            prev_keypoints[side]['elbow'] = elbow_pos
        return history

    def check_significant_movement(self, history):
        if len(history) < 2:
            return False
        total_movement = 0
        for i in range(len(history) - 1):
            curr = history[i]
            next_frame = history[i + 1]
            wrist_movement = np.sqrt((curr['wrist'][0] - next_frame['wrist'][0])**2 + (curr['wrist'][1] - next_frame['wrist'][1])**2)
            elbow_movement = np.sqrt((curr['elbow'][0] - next_frame['elbow'][0])**2 + (curr['elbow'][1] - next_frame['elbow'][1])**2)
            total_movement += (wrist_movement + elbow_movement) / 2
        avg_movement = total_movement / (len(history) - 1)
        return avg_movement > self.movement_threshold

    def update_and_check_movement(self, person_id, side, wrist_pos, elbow_pos, frame_count):
        tracker = self.get_tracker(person_id)
        history = self.calculate_keypoint_movement(person_id, side, wrist_pos, elbow_pos, frame_count)
        if frame_count % 3 == 0 and len(history) >= 2:
            if self.check_significant_movement(history):
                tracker['movement_counter'][side] += 1
                if tracker['movement_counter'][side] >= self.frame_memory:
                    tracker['action_active'][side] = True
            else:
                tracker['movement_counter'][side] = 0
                tracker['action_active'][side] = False
        return tracker['action_active'][side]

def calculate_iou_rotated(points1, points2):
    # Ensure points are in the correct format
    pts1 = np.array(points1, dtype=np.float32).reshape(-1, 2)  # Force 2D array
    pts2 = np.array(points2, dtype=np.float32).reshape(-1, 2)  # Force 2D array
    
    if len(pts1) < 3 or len(pts2) < 3:  # Need at least 3 points for a polygon
        return 0.0
        
    x_min = min(np.min(pts1[:, 0]), np.min(pts2[:, 0]))
    y_min = min(np.min(pts1[:, 1]), np.min(pts2[:, 1]))
    x_max = max(np.max(pts1[:, 0]), np.max(pts2[:, 0]))
    y_max = max(np.max(pts1[:, 1]), np.max(pts2[:, 1]))
    w = int((x_max - x_min))
    h = int((y_max - y_min))
    if w <= 0 or h <= 0:
        return 0.0
    
    # Create masks
    mask1 = np.zeros((h, w), dtype=np.uint8)
    mask2 = np.zeros((h, w), dtype=np.uint8)
    pts1_shifted = (pts1 - [x_min, y_min]).astype(np.int32)
    pts2_shifted = (pts2 - [x_min, y_min]).astype(np.int32)
    
    cv2.fillPoly(mask1, [pts1_shifted], 1)
    cv2.fillPoly(mask2, [pts2_shifted], 1)
    
    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    return intersection / union if union > 0 else 0.0

# Load models and ensure they stay on GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pose_model = YOLO("yolo11n-pose.pt").to(device)
seg_model = YOLOE("yoloe-11l-seg.pt").to(device)

# Set segmentation classes
seg_names = ["broom", "long thin broom", "sweeping tool", "floor cleaner", "mop"]
seg_model.set_classes(seg_names, seg_model.get_text_pe(seg_names))

# Video path
video_path = "/content/DT-036550-1.mp4"

# Initialize trackers and logs
movement_tracker = ArmMovementTracker(movement_threshold=7, frame_memory=15)
logging.basicConfig(filename="action_log.txt", level=logging.INFO, format="%(message)s")

# Load class labels
with open('class1.json', 'r') as f:
    class_data = json.load(f)
    categories = {cat['id']: cat['name'] for cat in class_data['categories']}

# Load object coordinates
boxes = []
with open('cord-1.txt', 'r') as f:
    for line in f:
        values = list(map(float, line.strip().split()))
        class_id = int(values[0])
        points = [(values[i], values[i + 1]) for i in range(1, len(values)-1, 2)]
        xs = [p[0] for p in points]
        ys = [p[1] for p in points]
        x_center = sum(xs) / len(xs)
        y_center = sum(ys) / len(ys)
        width = max(xs) - min(xs)
        height = max(ys) - min(ys)
        boxes.append({
            'class_id': class_id,
            'x_center': x_center,
            'y_center': y_center,
            'width': width,
            'height': height,
            'points': points,
            'label': categories.get(class_id, f"Class_{class_id}")
        })

# Process video
frame_count = 0

for pose_result, seg_result in zip(
    pose_model.track(video_path, stream=True, tracker="bytetrack.yaml", classes=[0], device=device),
    seg_model.track(video_path, conf=0.2, tracker="bytetrack.yaml", stream=True, device=device)
):
    frame_count += 1

    # Extract tracking info from pose_result
    tracked_people = {}
    if pose_result.boxes is not None and pose_result.boxes.id is not None:
        boxes_xyxy = pose_result.boxes.xyxy
        track_ids = pose_result.boxes.id
        for box, track_id in zip(boxes_xyxy, track_ids):
            x1, y1, x2, y2 = box.int().cpu().numpy()
            center = ((x1 + x2) / 2, (y1 + y2) / 2)
            tracked_people[int(track_id.cpu().item())] = {'box': (x1, y1, x2, y2), 'center': center}

    # Track objects in the current frame (segmentation)
    current_frame_seg_objects = []
    if seg_result.masks is not None and seg_result.boxes is not None:
        track_ids = seg_result.boxes.id if seg_result.boxes.id is not None else torch.tensor([], device=device)
        frame = seg_result.orig_img.copy()  # Get frame dimensions from original image
        for box, conf, cls, track_id, mask in zip(
            seg_result.boxes.xyxy,
            seg_result.boxes.conf,
            seg_result.boxes.cls,
            track_ids,
            seg_result.masks.data if seg_result.masks is not None else []
        ):
            label = seg_model.names[int(cls.cpu().item())]
            if not any(broom_term in label.lower() for broom_term in ["broom", "sweeping", "cleaner", "mop"]):
                continue
            x1, y1, x2, y2 = box.int().cpu().numpy()
            if mask is not None:
                mask_np = mask.cpu().numpy()
                mask_resized = cv2.resize(mask_np, (frame.shape[1], frame.shape[0]))
                contours, _ = cv2.findContours((mask_resized > 0.5).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                points = contours[0].reshape(-1, 2) if contours else np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]])
                current_frame_seg_objects.append({
                    'label': "broom",
                    'points': points,
                    'box': (x1, y1, x2, y2),
                    'track_id': int(track_id.cpu().item()) if track_id is not None else -1
                })

    seg_objects = current_frame_seg_objects

    # Process pose data
    if pose_result.keypoints is not None and len(pose_result.keypoints.data) > 0:
        track_to_pid = {}
        if pose_result.boxes is not None and pose_result.boxes.id is not None:
            sorted_track_ids = sorted([int(tid.cpu().item()) for tid in pose_result.boxes.id])
            for pid, track_id in enumerate(sorted_track_ids):
                track_to_pid[track_id] = pid

        current_seg_interactions = defaultdict(set)
        current_polygon_interactions = defaultdict(set)

        if pose_result.boxes is not None and pose_result.boxes.id is not None:
            for idx, (keypoints, track_id) in enumerate(zip(pose_result.keypoints.data, pose_result.boxes.id)):
                track_id = int(track_id.cpu().item())
                if track_id in tracked_people:
                    pid = track_to_pid[track_id]
                    keypoints_np = keypoints.cpu().numpy()

                    body_part_boxes = []
                    for arm_side, wrist_idx, elbow_idx in [('left', 9, 7), ('right', 10, 8)]:
                        if keypoints_np[wrist_idx][2] > 0.65 and keypoints_np[elbow_idx][2] > 0.65:
                            wrist = keypoints_np[wrist_idx][:2]
                            elbow = keypoints_np[elbow_idx][:2]
                            is_action_active = movement_tracker.update_and_check_movement(track_id, arm_side, wrist, elbow, frame_count)
                            min_x = min(wrist[0], elbow[0]) - 20
                            min_y = min(wrist[1], elbow[1]) - 20
                            max_x = max(wrist[0], elbow[0]) + 20
                            max_y = max(wrist[1], elbow[1]) + 20
                            arm_points = [
                                (min_x, min_y),
                                (max_x, min_y),
                                (max_x, max_y),
                                (min_x, max_y)
                            ]
                            body_part_boxes.append({
                                "box": [min_x, min_y, max_x, max_y],
                                "points": arm_points,
                                "name": f"P{pid}_{arm_side}_arm",
                                "hand": arm_side,
                                "person_id": track_id,
                                "display_id": pid
                            })

                    for box in boxes:
                        for body_part in body_part_boxes:
                            if keypoints_np[body_part['hand'] == 'left' and 9 or 10][2] > 0.5:
                                iou = calculate_iou_rotated(body_part["points"], box['points'])
                                if iou > 0.05:
                                    person_id = body_part['person_id']
                                    display_id = body_part['display_id']
                                    arm_side = body_part['hand']
                                    tracker = movement_tracker.get_tracker(person_id)
                                    if tracker['action_active'][arm_side]:
                                        current_polygon_interactions[display_id].add(box['label'])

                    for body_part in body_part_boxes:
                        for seg_obj in seg_objects:
                            if keypoints_np[body_part['hand'] == 'left' and 9 or 10][2] > 0.5:
                                iou = calculate_iou_rotated(body_part["points"], seg_obj['points'])
                                if iou > 0.05:
                                    person_id = body_part['person_id']
                                    display_id = body_part['display_id']
                                    arm_side = body_part['hand']
                                    tracker = movement_tracker.get_tracker(person_id)
                                    if tracker['action_active'][arm_side]:
                                        current_seg_interactions[display_id].add(seg_obj['label'])

        for display_id, interacted_objects in current_seg_interactions.items():
            for obj_label in interacted_objects:
                action = f"P{display_id} using {obj_label}"
                logging.info(f"Frame {frame_count}: {action} (segmentation)")

        for display_id, interacted_objects in current_polygon_interactions.items():
            for obj_label in interacted_objects:
                action = f"P{display_id} working with {obj_label}"
                logging.info(f"Frame {frame_count}: {action}, {len(tracked_people)} people")
    else:
        logging.info(f"Frame {frame_count}: No action detected, 0 people")